In [105]:
import cv2
import numpy as np
import time
from pathlib import Path

In [106]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = DATA_DIR / "echo.mp4"

## Few useful functions

color balance

In [107]:
def gray_world_balance(image):
  image_float = image.astype(np.float32)

  b, g, r = cv2.split(image_float)

  mean_b = np.mean(b)
  mean_g = np.mean(g)
  mean_r = np.mean(r)

  mean_gray = (mean_b + mean_g + mean_r) / 3.0

  eps = 1e-6

  b *= mean_gray / (mean_b + eps)
  g *= mean_gray / (mean_g + eps)
  r *= mean_gray / (mean_r + eps)

  balanced = cv2.merge([b, g, r])

  return np.clip(balanced, 0, 255).astype(np.uint8)

Log function

In [108]:
def log_transform(image):
  image_float = image.astype(np.float32)

  max_value = np.max(image_float)

  if max_value == 0:
      return np.zeros_like(image)

  c = 255.0 / np.log1p(max_value)

  transformed = (c * np.log1p(image_float))

  return np.clip(transformed, 0, 255).astype(np.uint8)

Gamma Function

In [109]:
def gamma_transform(image, gamma=1.2):
  normalized = (image.astype(np.float32) / 255.0)

  transformed = np.power(normalized, gamma)

  transformed *= 255.0

  return np.clip(transformed, 0, 255).astype(np.uint8)

full frame processing function

In [110]:
def process_frame(frame, gamma=1.2):

  # Convert BGR frame to grayscale
  gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

  # Improve global contrast
  equalized = cv2.equalizeHist(gray)

  # Convert intensity values into false colors
  heatmap = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)

  # Adjust channel balance
  balanced = gray_world_balance(heatmap)

  # Expand darker intensity variations
  log_frame = log_transform(balanced)

  # Suppress overly bright regions
  enhanced = gamma_transform(log_frame, gamma=gamma)

  return enhanced

Load

In [111]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

# MP4 codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

output_video_path = OUTPUT_DIR / "processed_echo.mp4"

if not cap.isOpened():
  raise FileNotFoundError(f"Could not open {VIDEO_PATH}")

Read vid info

In [112]:
fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Frames:", frame_count)

FPS: 50.0
Resolution: 112 x 112
Frames: 212


In [113]:
if fps > 0:
  duration = frame_count / fps
  print(f"Duration: {duration:.2f} seconds")

if fps <= 0:
  fps = 30

delay = max(1,int(1000 / fps))

Duration: 4.24 seconds


In [114]:
writer = cv2.VideoWriter(
    str(output_video_path),
    fourcc,
    fps,
    (width, height)
)

In [115]:
fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

output_path = "raw_vs_enhanced.mp4"

writer = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width * 2, height)
)

Final Loop

In [116]:
frame_number = 0

while True:
  ret, frame = cap.read()

  if not ret:
      break

  frame_number += 1

  start = time.perf_counter()

  enhanced = process_frame(frame, gamma=1.2)

  writer.write(enhanced)

  end = time.perf_counter()

  processing_time = end - start

  if processing_time > 0:
    processing_fps = 1 / processing_time
  else:
    processing_fps = 0

  raw_display = frame.copy()
  enhanced_display = enhanced.copy()

  cv2.putText(raw_display, "RAW", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

  cv2.putText(enhanced_display, "ENHANCED", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2
  )

  combined = np.hstack((raw_display, enhanced_display))

  cv2.putText(combined, f"Processing FPS: {processing_fps:.1f}", (20, combined.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255),2)

  #cv2.imshow("Echocardiogram Analysis", combined)

  #if(cv2.waitKey(delay) & 0xFF== ord("q")):
  #  break

  writer.write(combined)
cap.release()
cv2.destroyAllWindows()

print('Processing complete')

Processing complete


In [117]:
from IPython.display import Video

Video(
    "raw_vs_enhanced.mp4",
    embed=True
)

In [118]:
import os

path = "raw_vs_enhanced.mp4"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) if os.path.exists(path) else 0)

Exists: True
Size: 1310764


In [119]:
cap.release()
writer.release()
cv2.destroyAllWindows()

print(f"Saved processed video")

Saved processed video
